# 02 - Perfil de valores y limpieza

Perfila los valores de los archivos renombrados de beneficiarios FONASA, valida los totales por año, define reglas de limpieza para homologar categorías entre 2018 y 2025, las aplica a los 8 años y guarda los archivos limpios.

**Entrada:** `base-beneficiarios-data-renombrada/` - 8 archivos Parquet (`beneficiarios_fonasa_<año>.parquet`) generados por el notebook 01.

**Salida:** `base-beneficiarios-limpia/` - 8 archivos Parquet (`beneficiarios_fonasa_<año>_limpio.parquet`) con las columnas seleccionadas, categorías limpias y una columna `anio` agregada.

**Pasos**
1. Paso 1 - Listar los archivos Parquet renombrados
2. Paso 2 - Leer los nombres de columnas de cada archivo
3. Paso 3 - Seleccionar columnas presentes en al menos 6 de 8 años
4. Paso 4 - Perfilar valores por columna y año
   - Paso 4.1 - Construir la tabla de perfiles con value_counts
   - Paso 4.2 - Validar totales por año
5. Paso 5 - Revisar categorías por columna
   - Paso 5.1 - `sexo`
   - Paso 5.2 - `nacionalidad`
   - Paso 5.3 - `región_beneficiario`
   - Paso 5.4 - `comuna_beneficiario`
   - Paso 5.5 - `edad_tramo`, `titular_carga`, `tramo_fonasa`, `tipo_asegurado`, `mes_informacion`
   - Paso 5.6 - `caracterización` y `tramo_renta`
6. Paso 6 - Revisión exploratoria: mapeo de nombres de regiones
7. Paso 7 - Definir reglas de limpieza y probarlas sobre la tabla de perfiles
8. Paso 8 - Limpiar todos los años y guardar en Parquet
9. Paso 9 - Revisión final de los datos limpios
   - Paso 9.1 - Revisar etiquetas de comuna desconocida
   - Paso 9.2 - Revisar vacíos y categorías desconocidas

## Paso 1 - Listar los archivos Parquet renombrados

Lista los archivos Parquet escritos por el notebook 01 junto con su tamaño en MB, para confirmar que todos los años están disponibles antes de perfilar.

Resultado esperado: 8 archivos, de `beneficiarios_fonasa_2018.parquet` a `beneficiarios_fonasa_2025.parquet`, de entre aproximadamente 150 y 212 MB cada uno.

In [1]:
from pathlib import Path
import pandas as pd

carpeta_parquet = Path(r"C:\Users\ijoaq\PythonArchivos\PythonPractica\fondo-nacional-de-salud\base-beneficiarios-data-renombrada")

for archivo in sorted(carpeta_parquet.glob("*.parquet")):
    print(archivo.name, round(archivo.stat().st_size / 1e6, 1), "MB")

beneficiarios_fonasa_2018.parquet 198.9 MB
beneficiarios_fonasa_2019.parquet 207.9 MB
beneficiarios_fonasa_2020.parquet 211.6 MB
beneficiarios_fonasa_2021.parquet 149.7 MB
beneficiarios_fonasa_2022.parquet 179.9 MB
beneficiarios_fonasa_2023.parquet 190.0 MB
beneficiarios_fonasa_2024.parquet 200.0 MB
beneficiarios_fonasa_2025.parquet 209.6 MB


## Paso 2 - Leer los nombres de columnas de cada archivo

Imprime los nombres de columnas de cada archivo leyendo solo el esquema del Parquet, sin cargar datos.

Resultado esperado: 17 columnas en 2018-2020, 9 en 2021 y 12 en 2022-2025, todas con los nombres comunes definidos en el notebook 01.

In [2]:
import pyarrow.parquet as pq
for archivo in sorted(carpeta_parquet.glob("*parquet")):
    print(archivo.name, pq.read_schema(archivo).names)

beneficiarios_fonasa_2018.parquet ['mes_informacion', 'id_beneficiario', 'titular_carga', 'sexo', 'edad_tramo', 'cod_region_pernat', 'región_beneficiario', 'cod_comuna_pernat', 'comuna_beneficiario', 'nacionalidad', 'tramo_fonasa', 'caracterización', 'tipo_asegurado', 'servicio_salud', 'incrito_eaps', 'densidad', 'tramo_renta']
beneficiarios_fonasa_2019.parquet ['mes_informacion', 'id_beneficiario', 'titular_carga', 'sexo', 'edad_tramo', 'cod_region_pernat', 'región_beneficiario', 'cod_comuna_pernat', 'comuna_beneficiario', 'nacionalidad', 'tramo_fonasa', 'caracterización', 'tipo_asegurado', 'servicio_salud', 'incrito_eaps', 'densidad', 'tramo_renta']
beneficiarios_fonasa_2020.parquet ['mes_informacion', 'id_beneficiario', 'titular_carga', 'sexo', 'edad_tramo', 'cod_region_pernat', 'región_beneficiario', 'cod_comuna_pernat', 'comuna_beneficiario', 'nacionalidad', 'tramo_fonasa', 'caracterización', 'tipo_asegurado', 'servicio_salud', 'incrito_eaps', 'densidad', 'tramo_renta']
beneficiar

## Paso 3 - Seleccionar columnas presentes en al menos 6 de 8 años

Construye un crosstab de presencia de columnas por año a partir de los esquemas, agrega la cantidad de años en que aparece cada columna (`n_anios`) y guarda en `columnas` las que cumplen `n_anios >= 6`. Se descartan las columnas disponibles solo en 2018–2020.

Resultado esperado: 12 columnas seleccionadas. Nueve están presentes en los 8 años; `caracterización`, `tipo_asegurado` y `tramo_renta` faltan solo en 2021.

In [3]:
registros = []

for archivo in sorted(carpeta_parquet.glob("*.parquet")):
    anio = int(archivo.stem[-4:])
    for col in pq.read_schema(archivo).names:
        registros.append({"anio": anio, "columna": col})

registros = pd.DataFrame(registros)
presencia = pd.crosstab(registros["columna"], registros["anio"])
presencia["n_anios"] = presencia.sum(axis=1)

columnas = presencia[presencia["n_anios"] >= 6].index.tolist()
print(columnas)
presencia.sort_values("n_anios", ascending=False)

['caracterización', 'comuna_beneficiario', 'edad_tramo', 'id_beneficiario', 'mes_informacion', 'nacionalidad', 'región_beneficiario', 'sexo', 'tipo_asegurado', 'titular_carga', 'tramo_fonasa', 'tramo_renta']


anio,2018,2019,2020,2021,2022,2023,2024,2025,n_anios
columna,,,,,,,,,
comuna_beneficiario,1,1,1,1,1,1,1,1,8
tramo_fonasa,1,1,1,1,1,1,1,1,8
mes_informacion,1,1,1,1,1,1,1,1,8
id_beneficiario,1,1,1,1,1,1,1,1,8
edad_tramo,1,1,1,1,1,1,1,1,8
titular_carga,1,1,1,1,1,1,1,1,8
sexo,1,1,1,1,1,1,1,1,8
región_beneficiario,1,1,1,1,1,1,1,1,8
nacionalidad,1,1,1,1,1,1,1,1,8


## Paso 4 - Perfilar valores por columna y año

### Paso 4.1 - Construir la tabla de perfiles con value_counts

Para cada año carga solo las columnas seleccionadas (sin `id_beneficiario`) y calcula `value_counts(dropna=False)` por columna. Los resultados se apilan en una tabla larga `perfiles` (`valor`, `conteo`, `anio`, `columna`), lo bastante pequeña para revisar todas las categorías de todos los años.

Resultado esperado: una línea "Perfilando" por año y una tabla de 3.585 filas.

In [4]:
columnas_perfil = [c for c in columnas if c != "id_beneficiario"]

perfiles = []

for archivo in sorted(carpeta_parquet.glob("*.parquet")):
    anio = int(archivo.stem[-4:])
    disponibles = [c for c in columnas_perfil if c in pq.read_schema(archivo).names]
    print("Perfilando", anio)
    df = pd.read_parquet(archivo, columns=disponibles)

    for col in disponibles:
            conteo = df[col].value_counts(dropna=False).reset_index()
            conteo.columns = ["valor", "conteo"]
            conteo["anio"] = anio
            conteo["columna"] = col
            perfiles.append(conteo)

    del df

perfiles = pd.concat(perfiles, ignore_index=True)
perfiles

Perfilando

 2018


Perfilando 2019


Perfilando 2020


Perfilando 2021


Perfilando 2022


Perfilando 2023


Perfilando 2024


Perfilando 2025


,valor,conteo,anio,columna
0,DEPENDIENTE,6470138,2018,caracterización
1,SUF,1871565,2018,caracterización
2,PENSIONADO,1625467,2018,caracterización
3,GRUPO HOGAR CARENTE,1476961,2018,caracterización
4,NO DEFINIDO,1085887,2018,caracterización
...,...,...,...,...
3580,1.350.001 - 1.400.000,97807,2025,tramo_renta
3581,1.400.001 - 1.450.000,90615,2025,tramo_renta
3582,1.450.001 - 1.500.000,83784,2025,tramo_renta
3583,1.500.001 - 1.550.000,75317,2025,tramo_renta


### Paso 4.2 - Validar totales por año

Suma los conteos por año y columna. Todas las columnas de un mismo año deben sumar la misma cantidad de filas, y esos totales son los beneficiarios por año que se comparan con las cifras oficiales de FONASA.

Resultado esperado: totales idénticos entre columnas dentro de cada año: 14.102.709 (2018), 14.841.577 (2019), 15.142.528 (2020), 15.233.814 (2021), 15.613.584 (2022), 16.229.898 (2023), 16.752.189 (2024), 17.132.611 (2025). `caracterización`, `tipo_asegurado` y `tramo_renta` aparecen como NaN en 2021 porque no existen en ese archivo.

In [5]:
perfiles.groupby(["anio", "columna"])["conteo"].sum().unstack()

columna,caracterización,comuna_beneficiario,edad_tramo,mes_informacion,nacionalidad,región_beneficiario,sexo,tipo_asegurado,titular_carga,tramo_fonasa,tramo_renta
anio,,,,,,,,,,,
2018,14102709.0,14102709.0,14102709.0,14102709.0,14102709.0,14102709.0,14102709.0,14102709.0,14102709.0,14102709.0,14102709.0
2019,14841577.0,14841577.0,14841577.0,14841577.0,14841577.0,14841577.0,14841577.0,14841577.0,14841577.0,14841577.0,14841577.0
2020,15142528.0,15142528.0,15142528.0,15142528.0,15142528.0,15142528.0,15142528.0,15142528.0,15142528.0,15142528.0,15142528.0
2021,NaN,15233814.0,15233814.0,15233814.0,15233814.0,15233814.0,15233814.0,NaN,15233814.0,15233814.0,NaN
2022,15613584.0,15613584.0,15613584.0,15613584.0,15613584.0,15613584.0,15613584.0,15613584.0,15613584.0,15613584.0,15613584.0
2023,16229898.0,16229898.0,16229898.0,16229898.0,16229898.0,16229898.0,16229898.0,16229898.0,16229898.0,16229898.0,16229898.0
2024,16752189.0,16752189.0,16752189.0,16752189.0,16752189.0,16752189.0,16752189.0,16752189.0,16752189.0,16752189.0,16752189.0
2025,17132611.0,17132611.0,17132611.0,17132611.0,17132611.0,17132611.0,17132611.0,17132611.0,17132611.0,17132611.0,17132611.0


## Paso 5 - Revisar categorías por columna

### Paso 5.1 - `sexo`

Muestra el perfil de `sexo` como tabla dinámica de valor por año para comparar etiquetas entre períodos. Se ven etiquetas en mayúscula en 2018-2020, formato título desde 2021, y grupos pequeños "Indeterminado" / "Sin información" que cambian de nombre entre años.

In [6]:
perfiles[perfiles["columna"] == "sexo"].pivot_table(index="valor", columns="anio", values="conteo")

anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
HOMBRE,6609609.0,6994917.0,7189752.0,NaN,NaN,NaN,NaN,NaN
Hombre,NaN,NaN,NaN,7246170.0,7432827.0,7744672.0,8015233.0,8227554.0
INDETERMINADO,147.0,345.0,414.0,NaN,NaN,NaN,NaN,NaN
Indeterminado,NaN,NaN,NaN,543.0,NaN,NaN,1246.0,1471.0
MUJER,7492953.0,7846315.0,7952362.0,NaN,NaN,NaN,NaN,NaN
Mujer,NaN,NaN,NaN,7987101.0,8180051.0,8484218.0,8735710.0,8903586.0
Sin información,NaN,NaN,NaN,NaN,706.0,1008.0,NaN,NaN


### Paso 5.2 - `nacionalidad`

Muestra el perfil de `nacionalidad` por año. La única diferencia entre años es el uso de mayúsculas (`CHILENA` / `Chilena`, `EXTRANJERA` / `Extranjera`).

In [7]:
perfiles[perfiles["columna"] == "nacionalidad"].pivot_table(index="valor", columns="anio", values="conteo")

anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
CHILENA,13426296.0,13815819.0,14060709.0,NaN,NaN,NaN,NaN,NaN
Chilena,NaN,NaN,NaN,14037847.0,14216370.0,14676433.0,15004523.0,15222189.0
EXTRANJERA,676413.0,1025758.0,1081819.0,NaN,NaN,NaN,NaN,NaN
Extranjera,NaN,NaN,NaN,1195967.0,1397214.0,1553465.0,1747666.0,1910422.0


### Paso 5.3 - `región_beneficiario`

Muestra el perfil de `región_beneficiario` por año. Los nombres de región varían en mayúsculas y redacción entre años (por ejemplo "DEL LIBERTADOR B. O'HIGGINS" frente al nombre completo), por lo que hay que llevarlos a los nombres oficiales.

In [8]:
perfiles[perfiles["columna"] == "región_beneficiario"].pivot_table(index="valor", columns="anio", values="conteo")

anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
DE ANTOFAGASTA,413731.0,442613.0,457944.0,NaN,NaN,NaN,NaN,NaN
DE ARICA Y PARINACOTA,189270.0,200890.0,205787.0,NaN,NaN,NaN,NaN,NaN
DE ATACAMA,240935.0,252071.0,257377.0,NaN,NaN,NaN,NaN,NaN
DE AYSÉN DEL GRAL. C. IBÁÑEZ DEL CAMPO,75617.0,79501.0,81125.0,NaN,NaN,NaN,NaN,NaN
DE COQUIMBO,648137.0,683248.0,699935.0,NaN,NaN,NaN,NaN,NaN
DE LA ARAUCANÍA,844771.0,879389.0,899453.0,NaN,NaN,NaN,NaN,NaN
DE LOS LAGOS,737343.0,767127.0,785248.0,NaN,NaN,NaN,NaN,NaN
DE LOS RÍOS,340809.0,353091.0,358794.0,NaN,NaN,NaN,NaN,NaN
DE MAGALLANES Y DE LA ANTÁRTICA CHILENA,122885.0,129571.0,132767.0,NaN,NaN,NaN,NaN,NaN


### Paso 5.4 - `comuna_beneficiario`

Pasa los nombres de comuna a formato título y deja solo las comunas que faltan en al menos un año. Tras el cambio de formato, las diferencias que quedan son variantes de escritura de una misma comuna.

Resultado esperado: solo "Cabo De Hornos" / "Cabo De Hornos (Ex - Navarino)" y "Ranquil" / "Ránquil".

In [9]:
comunas = perfiles[perfiles["columna"] == "comuna_beneficiario"].copy()
comunas["valor"] = comunas["valor"].str.title()

presencia_comunas = comunas.pivot_table(index="valor", columns="anio", values="conteo", aggfunc="sum")
presencia_comunas[presencia_comunas.isna().any(axis=1)]

anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
Cabo De Hornos,331.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cabo De Hornos (Ex - Navarino),NaN,362.0,359.0,361.0,377.0,373.0,283.0,612.0
Ranquil,NaN,NaN,NaN,5929.0,5992.0,6156.0,6242.0,6322.0
Ránquil,5455.0,5773.0,5840.0,NaN,NaN,NaN,NaN,NaN


### Paso 5.5 - `edad_tramo`, `titular_carga`, `tramo_fonasa`, `tipo_asegurado`, `mes_informacion`

Muestra la tabla dinámica de valor por año de cada una de estas columnas. Se ve que los tramos de edad cambian de amplitud entre años (2 a 5 años hasta 2021, 10 años desde 2022), que `titular_carga` y `tipo_asegurado` difieren en mayúsculas y redacción, que `tramo_fonasa` ya es consistente (A-D) y que `mes_informacion` tiene un único corte de diciembre por año (`201812` … `202512`).

In [10]:
for col in ["edad_tramo", "titular_carga", "tramo_fonasa", "tipo_asegurado", "mes_informacion"]:
    print(col)
    display(perfiles[perfiles["columna"] == col].pivot_table(index="valor", columns="anio", values="conteo"))

edad_tramo


anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
0 a 9 años,NaN,NaN,NaN,NaN,1752272.0,1764951.0,NaN,NaN
00 a 02 años,497238.0,505899.0,485171.0,NaN,NaN,NaN,NaN,400291.0
00 a 04 años,NaN,NaN,NaN,789561.0,NaN,NaN,NaN,NaN
00 a 4 años,NaN,NaN,NaN,NaN,NaN,NaN,750783.0,NaN
03 a 04 años,377943.0,372767.0,353354.0,NaN,NaN,NaN,NaN,320386.0
05 a 09 años,960934.0,979078.0,972109.0,957246.0,NaN,NaN,998137.0,973108.0
10 a 14 años,925407.0,962753.0,984405.0,997310.0,NaN,NaN,1104422.0,1120865.0
10 a 19 años,NaN,NaN,NaN,NaN,2003855.0,2094926.0,NaN,NaN
15 a 19 años,972362.0,970703.0,960580.0,957076.0,NaN,NaN,1083894.0,1123193.0


titular_carga


anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
CARGA,2905988.0,3004162.0,3061977.0,NaN,NaN,NaN,NaN,NaN
Carga,NaN,NaN,NaN,2967185.0,3001047.0,3142295.0,3240256.0,3111232.0
TITULAR,11196721.0,11837415.0,12080551.0,NaN,NaN,NaN,NaN,NaN
Titular,NaN,NaN,NaN,12266629.0,12612537.0,13087603.0,13511933.0,14021379.0


tramo_fonasa


anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
A,3347167.0,3272101.0,3108099.0,2999241.0,3061800.0,3041989.0,3157865.0,3074735.0
B,4971311.0,5261606.0,5921159.0,6001917.0,6155357.0,6596229.0,6761416.0,7002990.0
C,2272917.0,2408784.0,2490305.0,2370547.0,2544949.0,2656732.0,2790265.0,2541272.0
D,3511314.0,3899086.0,3622965.0,3862109.0,3851478.0,3934948.0,4042643.0,4513614.0


tipo_asegurado


anio,2018,2019,2020,2022,2023,2024,2025
valor,,,,,,,
BLOQUEO,38.0,NaN,NaN,NaN,NaN,NaN,NaN
CARENTE,3339716.0,3267916.0,3105233.0,NaN,NaN,NaN,NaN
Carente,NaN,NaN,NaN,3060138.0,3040763.0,3157459.0,NaN
Carente Recursos,NaN,NaN,NaN,NaN,NaN,NaN,3074357.0
DESCONOCIDO,60.0,3112.0,14.0,NaN,NaN,NaN,NaN
Dependiente,NaN,NaN,NaN,NaN,NaN,NaN,9833995.0
Desconocido,NaN,NaN,NaN,15.0,22.0,3724.0,2003.0
Independiente,NaN,NaN,NaN,NaN,NaN,NaN,1345624.0
PENSIONADO,2173436.0,2111138.0,2054466.0,NaN,NaN,NaN,NaN


mes_informacion


anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
201812,14102709.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201912,NaN,14841577.0,NaN,NaN,NaN,NaN,NaN,NaN
202012,NaN,NaN,15142528.0,NaN,NaN,NaN,NaN,NaN
202112,NaN,NaN,NaN,15233814.0,NaN,NaN,NaN,NaN
202212,NaN,NaN,NaN,NaN,15613584.0,NaN,NaN,NaN
202312,NaN,NaN,NaN,NaN,NaN,16229898.0,NaN,NaN
202412,NaN,NaN,NaN,NaN,NaN,NaN,16752189.0,NaN
202512,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17132611.0


### Paso 5.6 - `caracterización` y `tramo_renta`

Muestra la tabla dinámica de valor por año de estas dos columnas después de pasar los valores a formato título. Ambas tienen muchas categorías equivalentes con nombres distintos, y `tramo_renta` usa tramos de $50.000 en algunos años y de $250.000 en otros, por lo que las dos necesitan mapeos explícitos.

In [11]:
for col in ["caracterización", "tramo_renta"]:
    tabla = perfiles[perfiles["columna"] == col].copy()
    tabla["valor"] = tabla["valor"].str.title()
    print(col)
    display(tabla.pivot_table(index="valor", columns="anio", values="conteo", aggfunc="sum"))

caracterización


anio,2018,2019,2020,2022,2023,2024,2025
valor,,,,,,,
Afiliación Extendida Por Licencia Médica,NaN,30.0,95.0,179.0,308.0,493.0,330.0
Bloqueo,148.0,231714.0,191229.0,NaN,NaN,NaN,NaN
Carente De Recursos Grupo Hogar,NaN,NaN,NaN,NaN,NaN,NaN,1587956.0
Chile-España,113.0,145.0,160.0,222.0,248.0,279.0,NaN
Convenio Chile-España,NaN,NaN,NaN,NaN,NaN,NaN,60.0
Dependiente,6470138.0,8067251.0,8548769.0,8204155.0,7884598.0,9189348.0,8490416.0
Desempleado Con Seguro De Desempleo,38605.0,41190.0,46175.0,906017.0,1345078.0,687756.0,NaN
Empleado Del Sector Salud,258283.0,255762.0,274374.0,297215.0,288407.0,300321.0,NaN
Extensión Trabajo Temporal,2307.0,2589.0,2869.0,1768.0,1303.0,656.0,NaN


tramo_renta


anio,2018,2019,2020,2022,2023,2024,2025
valor,,,,,,,
1 - 100.000,324936.0,300200.0,270627.0,209939.0,190720.0,NaN,166970.0
1 A 249.999,NaN,NaN,NaN,NaN,NaN,1466532.0,NaN
1.000.000 A 1.249.999,NaN,NaN,NaN,NaN,NaN,692085.0,NaN
1.000.001 - 1.050.000,62359.0,72926.0,80853.0,116318.0,150331.0,NaN,214925.0
1.050.001 - 1.100.000,52530.0,62433.0,69080.0,102534.0,134879.0,NaN,181011.0
1.100.001 - 1.150.000,44246.0,52943.0,59974.0,89708.0,119197.0,NaN,161220.0
1.150.001 - 1.200.000,37267.0,44134.0,50989.0,79253.0,107260.0,NaN,142818.0
1.200.001 - 1.250.000,31381.0,37444.0,44157.0,70037.0,96532.0,NaN,130016.0
1.250.000 A 1.499.999,NaN,NaN,NaN,NaN,NaN,419464.0,NaN


## Paso 6 - Revisión exploratoria: mapeo de nombres de regiones

Primera prueba de un mapeo de regiones: normaliza `región_beneficiario` en la tabla de perfiles (strip + formato título) y reemplaza las variantes por los nombres oficiales de región. Esta versión de `REGIONES` queda reemplazada por la que se define en el Paso 7.

Resultado esperado: 16 nombres oficiales de región más "Desconocida", sin regiones duplicadas entre años.

In [12]:
def normalizar(serie):
    return serie.str.strip().str.title()

REGIONES = {
    "De Arica Y Parinacota": "Región de Arica y Parinacota",
    "De Tarapacá": "Región de Tarapacá",
    "De Antofagasta": "Región de Antofagasta",
    "De Atacama": "Región de Atacama",
    "De Coquimbo": "Región de Coquimbo",
    "De Valparaíso": "Región de Valparaíso",
    "Metropolitana De Santiago": "Región Metropolitana de Santiago",
    "Del Libertador General Bernardo O'Higgins": "Región del Libertador General Bernardo O'Higgins",
    "Del Libertador B. O'Higgins": "Región del Libertador General Bernardo O'Higgins",
    "Del Maule": "Región del Maule",
    "Ñuble": "Región de Ñuble",
    "De Ñuble": "Región de Ñuble",
    "Del Bíobío": "Región del Biobío",
    "De La Araucanía": "Región de La Araucanía",
    "De Los Ríos": "Región de Los Ríos",
    "De Los Lagos": "Región de Los Lagos",
    "De Aysén Del Gral. C. Ibáñez Del Campo": "Región Aysén del General Carlos Ibáñez del Campo",
    "De Magallanes Y De La Antártica Chilena": "Región de Magallanes y de la Antártica Chilena",
}

prueba = perfiles[perfiles["columna"] == "región_beneficiario"].copy()
prueba["valor"] = normalizar(prueba["valor"]).replace(REGIONES)
prueba.pivot_table(index="valor", columns="anio", values="conteo", aggfunc="sum")

anio,2018,2019,2020,2021,2022,2023,2024,2025
valor,,,,,,,,
Desconocida,202488,212817,228169,195187,165720,171644,191087,180139
Región Aysén del General Carlos Ibáñez del Campo,75617,79501,81125,81345,82415,84083,83853,85129
Región Metropolitana de Santiago,5120290,5416715,5508343,5542187,5680677,5945628,6135757,6271921
Región de Antofagasta,413731,442613,457944,469584,495329,527664,553748,581007
Región de Arica y Parinacota,189270,200890,205787,212774,223720,232060,242044,249435
Región de Atacama,240935,252071,257377,259928,268335,280270,290037,297255
Región de Coquimbo,648137,683248,699935,707781,725960,750485,770960,788948
Región de La Araucanía,844771,879389,899453,908737,930651,957860,984188,1004831
Región de Los Lagos,737343,767127,785248,788939,807545,837221,865652,885369


## Limpieza de datos

### Alcance
- Base de beneficiarios FONASA (corte diciembre) entre 2018 y 2025.
- Se conservaron solo las columnas presentes en al menos 6 de los 8 años.
- Algunas variables no están disponibles en todos los períodos (especialmente en 2021).

### Transformaciones principales
- Normalización de texto (espacios, mayúsculas/minúsculas y nombres inconsistentes).
- Corrección y unificación de nombres de regiones y comunas.
- Agrupación de edades en tramos de 10 años.
- Homologación de categorías de `tipo_asegurado` y `caracterización`.
- Agrupación de `tramo_renta` en bandas de CLP 250.000.
- Valores equivalentes o poco frecuentes fueron fusionados en categorías comunes.

### Supuestos
- Categorías con distinto nombre pero mismo significado fueron unificadas.
- Pequeñas diferencias en límites de tramos de renta se consideraron equivalentes.

### Consideraciones
- Existen posibles cambios metodológicos en algunas variables durante 2025.
- `caracterización` presenta problemas de calidad en los primeros años.
- Los tramos de renta están expresados en pesos nominales (sin ajuste por inflación).
- Los totales territoriales pueden diferir de otras fuentes que excluyen registros sin ubicación conocida.

## Paso 7 - Definir reglas de limpieza y probarlas sobre la tabla de perfiles

Define las funciones y diccionarios de limpieza resumidos en la celda "Limpieza de datos" anterior: normalización de texto (`estandarizar`), nombres oficiales de regiones, correcciones de comunas, tramos de edad de 10 años, tramos de renta de $250.000 y categorías unificadas para `sexo`, `tipo_asegurado` y `caracterización`. `limpiar_df` las aplica a un DataFrame completo, rellena región y comuna vacías con "Desconocida" y agrega la columna `anio`. Las reglas se prueban primero sobre `perfiles` para comprobar que solo renombran categorías y no cambian los conteos.

Resultado esperado: 10 tramos de edad, 8 tramos de renta, 17 regiones, 347 comunas, 5 categorías de `tipo_asegurado` y 26 de `caracterización`, y "Totales iguales: True".

In [13]:
def invertir(grupos):
    return {variante: destino for destino, variantes in grupos.items() for variante in variantes}


def estandarizar(valor, reemplazos=None, title=True):
    v = valor.strip()
    if title:
        v = v.title()
    if reemplazos:
        v = reemplazos.get(v, v)
    return v


SEXO = invertir({
    "Indeterminado o sin información": ["Indeterminado", "Sin Información"],
})

REGIONES = invertir({
    "Región de Arica y Parinacota": ["De Arica Y Parinacota"],
    "Región de Tarapacá": ["De Tarapacá"],
    "Región de Antofagasta": ["De Antofagasta"],
    "Región de Atacama": ["De Atacama"],
    "Región de Coquimbo": ["De Coquimbo"],
    "Región de Valparaíso": ["De Valparaíso"],
    "Región Metropolitana de Santiago": ["Metropolitana De Santiago"],
    "Región del Libertador General Bernardo O'Higgins": ["Del Libertador General Bernardo O'Higgins", "Del Libertador B. O'Higgins"],
    "Región del Maule": ["Del Maule"],
    "Región de Ñuble": ["Ñuble", "De Ñuble"],
    "Región del Biobío": ["Del Bíobío"],
    "Región de La Araucanía": ["De La Araucanía"],
    "Región de Los Ríos": ["De Los Ríos"],
    "Región de Los Lagos": ["De Los Lagos"],
    "Región Aysén del General Carlos Ibáñez del Campo": ["De Aysén Del Gral. C. Ibáñez Del Campo"],
    "Región de Magallanes y de la Antártica Chilena": ["De Magallanes Y De La Antártica Chilena"],
})

COMUNAS = {
    "Cabo De Hornos (Ex - Navarino)": "Cabo De Hornos",
    "Ranquil": "Ránquil",
}

CONECTORES = {"De", "Del", "Y"}
ARTICULOS = {"La", "Las", "Los", "El"}


def ajustar_conectores(nombre):
    palabras = nombre.split(" ")
    for i in range(1, len(palabras)):
        if palabras[i] in CONECTORES:
            palabras[i] = palabras[i].lower()
        elif palabras[i] in ARTICULOS and palabras[i - 1] in ("de", "del"):
            palabras[i] = palabras[i].lower()
    return " ".join(palabras)


def limpiar_comuna(valor):
    v = estandarizar(valor, COMUNAS)
    v = v.replace("Bíobío", "Biobío")
    return ajustar_conectores(v)


EDAD = invertir({
    "00 a 09 años": ["0 a 9 años", "00 a 02 años", "00 a 04 años", "00 a 4 años", "03 a 04 años", "05 a 09 años"],
    "10 a 19 años": ["10 a 14 años", "10 a 19 años", "15 a 19 años"],
    "20 a 29 años": ["20 a 24 años", "20 a 29 años", "25 a 29 años"],
    "30 a 39 años": ["30 a 34 años", "30 a 39 años", "35 a 39 años"],
    "40 a 49 años": ["40 a 44 años", "40 a 49 años", "45 a 49 años"],
    "50 a 59 años": ["50 a 54 años", "50 a 59 años", "55 a 59 años"],
    "60 a 69 años": ["60 a 64 años", "60 a 69 años", "65 a 69 años"],
    "70 a 79 años": ["70 a 74 años", "70 a 79 años", "75 a 79 años"],
    "80 años o más": ["80 a 84 años", "80 a más años", "85 a 89 años", "90 a 94 años", "95 a 99 años", "Más de 99 años"],
    "Sin información": ["S.I.", "Sin información"],
})

TIPO_ASEGURADO = invertir({
    "Trabajador dependiente": ["Trabajador Dependiente", "Dependiente"],
    "Trabajador independiente": ["Trabajador Independiente", "Independiente"],
    "Carente de recursos": ["Carente", "Carente Recursos"],
    "Pensionado": ["Pensionado"],
    "Desconocido": ["Desconocido", "Bloqueo"],
})

CARACTERIZACION = invertir({
    "Afiliación extendida por licencia médica": ["Afiliación Extendida Por Licencia Médica"],
    "Bloqueo": ["Bloqueo"],
    "Carente de recursos (grupo hogar)": ["Grupo Hogar Carente", "Carente De Recursos Grupo Hogar"],
    "Convenio Chile-España": ["Chile-España", "Convenio Chile-España"],
    "Dependiente": ["Dependiente"],
    "Desempleado con seguro de desempleo": ["Desempleado Con Seguro De Desempleo", "Seguro De Desempleo"],
    "Funcionario del sector salud": ["Empleado Del Sector Salud", "Funcionario Del Sector Salud"],
    "Temporero obra o faena (afiliación extendida)": ["Extensión Trabajo Temporal", "Trabajador Temporero Obra O Faena Con Afiliación Extendida"],
    "Temporero turno o jornada (afiliación extendida)": ["Temporero (Turno O Jornada) Afiliación Ext", "Trabajador Temporero Turno O Jornada Con Afiliacion Extendida"],
    "Temporero con afiliación extendida": ["Temporero Con Afiliación Extendida"],
    "Extranjero con visa en trámite": ["Extranjero Visa En Trámite"],
    "Independiente": ["Independiente"],
    "Independiente voluntario": ["Independiente Voluntario"],
    "Menor Sename": ["Sename", "Menor Sename"],
    "No definido": ["No Definido", "No Caracterizado"],
    "Orfandad": ["Orfandad"],
    "PBS sin cotizaciones (cobertura 100% MAI)": ["Pbs Cobertura 100% Mai", "Pbs Cobertura 100%, Prestación Mai", "Pbs Sin Cotizaciones Cobertura 100% Mai"],
    "PBS con cotización": ["Pbs Con Cotizaciones", "Pbs Con Cotización"],
    "Pensionado": ["Pensionado"],
    "Pensionado del sector salud": ["Pensionado Sector Salud", "Pensionado Del Sector Salud"],
    "Pensionado extranjero": ["Pensionado Extranjero"],
    "Pensión en trámite": ["Pensión En Proceso", "Pensión En Trámite"],
    "Subsidio de cesantía": ["Receptor De Subsidio De Cesantía", "Subsidio De Cesantía"],
    "Recién contratado": ["Recién Contratado"],
    "SUF": ["Suf"],
    "Trabajador a honorarios": ["Trabajador A Honorario"],
})

RENTA = invertir({
    "1 a 249.999": ["1 - 100.000", "100.001 - 150.000", "150.001 - 200.000", "200.001 - 250.000", "1 A 249.999"],
    "250.000 a 499.999": ["250.001 - 300.000", "300.001 - 350.000", "350.001 - 400.000", "400.001 - 450.000", "450.001 - 500.000", "250.000 A 499.999"],
    "500.000 a 749.999": ["500.001 - 550.000", "550.001 - 600.000", "600.001 - 650.000", "650.001 - 700.000", "700.001 - 750.000", "500.000 A 749.999"],
    "750.000 a 999.999": ["750.001 - 800.000", "800.001 - 850.000", "850.001 - 900.000", "900.001 - 950.000", "950.001 - 1.000.000", "750.000 A 999.999"],
    "1.000.000 a 1.249.999": ["1.000.001 - 1.050.000", "1.050.001 - 1.100.000", "1.100.001 - 1.150.000", "1.150.001 - 1.200.000", "1.200.001 - 1.250.000", "1.000.000 A 1.249.999"],
    "1.250.000 a 1.499.999": ["1.250.001 - 1.300.000", "1.300.001 - 1.350.000", "1.350.001 - 1.400.000", "1.400.001 - 1.450.000", "1.450.001 - 1.500.000", "1.250.000 A 1.499.999"],
    "1.500.000 o más": ["1.500.001 - 1.550.000", "1.550.001 - 1.600.000", "Mayor A 1.600.00",
                        "1.500.000 A 1.749.999", "1.750.000 A 1.999.999", "2.000.000 A 2.249.999", "2.250.000 A 2.499.999",
                        "2.500.000 A 2.749.999", "2.750.000 A 2.999.999", "Mayor A 3.000.000"],
    "Sin cotizaciones": ["Sin Cotizaciones", "Pbs Sin Cotizaciones", "Sin Información"],
})

LIMPIEZA = {
    "sexo": lambda v: estandarizar(v, SEXO),
    "nacionalidad": lambda v: estandarizar(v),
    "titular_carga": lambda v: estandarizar(v),
    "tramo_fonasa": lambda v: estandarizar(v),
    "región_beneficiario": lambda v: estandarizar(v, REGIONES),
    "comuna_beneficiario": limpiar_comuna,
    "edad_tramo": lambda v: estandarizar(v, EDAD, title=False),
    "tipo_asegurado": lambda v: estandarizar(v, TIPO_ASEGURADO),
    "caracterización": lambda v: estandarizar(v, CARACTERIZACION),
    "tramo_renta": lambda v: estandarizar(v, RENTA),
}


def limpiar_df(df, anio):
    for col, funcion in LIMPIEZA.items():
        if col in df.columns:
            mapa = {v: funcion(v) for v in df[col].dropna().unique()}
            df[col] = df[col].map(mapa)
    df["comuna_beneficiario"] = df["comuna_beneficiario"].fillna("Desconocida")
    df["región_beneficiario"] = df["región_beneficiario"].fillna("Desconocida")
    df["anio"] = anio
    return df


prueba = perfiles.copy()
for col, funcion in LIMPIEZA.items():
    filtro = prueba["columna"] == col
    valores = prueba.loc[filtro, "valor"]
    mapa = {v: funcion(v) for v in valores.dropna().unique()}
    prueba.loc[filtro, "valor"] = valores.map(mapa)

print(prueba.groupby("columna")["valor"].nunique())

totales_antes = perfiles.groupby(["anio", "columna"])["conteo"].sum()
totales_despues = prueba.groupby(["anio", "columna"])["conteo"].sum()
print("Totales iguales:", totales_antes.equals(totales_despues))

columna
caracterización         26
comuna_beneficiario    347
edad_tramo              10
mes_informacion          8
nacionalidad             2
región_beneficiario     17
sexo                     3
tipo_asegurado           5
titular_carga            2
tramo_fonasa             4
tramo_renta              8
Name: valor, dtype: int64
Totales iguales: True


## Paso 8 - Limpiar todos los años y guardar en Parquet

Carga las columnas seleccionadas de cada año, aplica `limpiar_df` y escribe el resultado en `base-beneficiarios-limpia/` como `beneficiarios_fonasa_<año>_limpio.parquet`. Cada DataFrame se elimina después de guardarlo para limitar el uso de memoria.

Resultado esperado: la cantidad de filas impresa por año coincide con los totales del Paso 4.2 (14.102.709 en 2018 hasta 17.132.611 en 2025), seguida de "Listo".

In [14]:
carpeta_limpia = carpeta_parquet.parent / "base-beneficiarios-limpia"
carpeta_limpia.mkdir(exist_ok=True)

for archivo in sorted(carpeta_parquet.glob("*.parquet")):
    anio = int(archivo.stem[-4:])
    disponibles = [c for c in columnas if c in pq.read_schema(archivo).names]
    df = pd.read_parquet(archivo, columns=disponibles)
    df = limpiar_df(df, anio)
    print("Limpiando", anio, "-", len(df), "filas")
    df.to_parquet(carpeta_limpia / f"beneficiarios_fonasa_{anio}_limpio.parquet", index=False)
    del df

print("Listo")

Limpiando 2018 - 14102709 filas


Limpiando 2019 - 14841577 filas


Limpiando 2020 - 15142528 filas


Limpiando 2021 - 15233814 filas


Limpiando 2022 - 15613584 filas


Limpiando 2023 - 16229898 filas


Limpiando 2024 - 16752189 filas


Limpiando 2025 - 17132611 filas


Listo


## Paso 9 - Revisión final de los datos limpios

### Paso 9.1 - Revisar etiquetas de comuna desconocida

Busca en los valores de comuna ya limpios de la tabla de prueba (`prueba`, del Paso 7) cualquier etiqueta que contenga "esconoc" o "Sin", para confirmar que las 347 comunas son 346 comunas reales más una única etiqueta de desconocida.

Resultado esperado: `['Desconocida']`.

In [15]:
[c for c in prueba.loc[prueba["columna"] == "comuna_beneficiario", "valor"].dropna().unique() if "esconoc" in c or "Sin" in c]

['Desconocida']

### Paso 9.2 - Revisar vacíos y categorías desconocidas

Lee cada archivo limpio y calcula, por columna y año, la cantidad de vacíos y el porcentaje de valores etiquetados como desconocidos ("Desconocida", "Desconocido", "Sin información", "Indeterminado o sin información", "No definido").

Resultado esperado: 0 vacíos en todas las columnas (NaN solo en las columnas que no existen en 2021). Los porcentajes de desconocidos son bajos, cerca de 1-1,5% en región y comuna; el más alto es `caracterización` en 2018 (7,70%).

In [16]:
import re
DESCONOCIDOS = ["Desconocida", "Desconocido", "Sin información", "Indeterminado o sin información", "No definido"]

resumen = []

for archivo in sorted(carpeta_limpia.glob("*.parquet")):
    columnas = [c for c in pq.read_schema(archivo).names if c not in ("id_beneficiario", "anio")]
    df = pd.read_parquet(archivo, columns=columnas)
    anio = int(re.search(r"(19|20)\d{2}", archivo.stem).group())
    for col in columnas:
        resumen.append({
            "anio": anio,
            "columna": col,
            "vacios": df[col].isna().sum(),
            "pct_desconocidos": round(df[col].isin(DESCONOCIDOS).mean() * 100, 2),
        })
    del df

resumen = pd.DataFrame(resumen)

print("Vacíos")
display(resumen.pivot_table(index="columna", columns="anio", values="vacios"))
print("% desconocidos")
display(resumen.pivot_table(index="columna", columns="anio", values="pct_desconocidos"))

Vacíos


anio,2018,2019,2020,2021,2022,2023,2024,2025
columna,,,,,,,,
caracterización,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0
comuna_beneficiario,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
edad_tramo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mes_informacion,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
nacionalidad,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
región_beneficiario,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
sexo,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
tipo_asegurado,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0
titular_carga,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


% desconocidos


anio,2018,2019,2020,2021,2022,2023,2024,2025
columna,,,,,,,,
caracterización,7.70,0.03,0.01,NaN,0.02,0.08,0.16,0.32
comuna_beneficiario,1.44,1.43,1.51,1.28,1.06,1.06,1.15,1.05
edad_tramo,0.04,0.04,0.04,0.05,0.05,0.08,0.08,0.08
mes_informacion,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
nacionalidad,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
región_beneficiario,1.44,1.43,1.51,1.28,1.06,1.06,1.15,1.05
sexo,0.00,0.00,0.00,0.00,0.00,0.01,0.01,0.01
tipo_asegurado,0.00,0.02,0.00,NaN,0.00,0.00,0.02,0.01
titular_carga,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
